In [2]:
import tarfile
import pandas as pd
import os

with tarfile.open('misinfo-resources-2021.tar.gz', 'r:gz') as tar:
    tar.extractall('trec_health_eval/')
print("📁 Contents:")
for file in os.listdir('trec_health_eval/'):
    print(f"  {file}")

📁 Contents:
  misinfo-resources-2021


### 1. QRELS (credibility gold standard)

In [3]:
qrels_df = pd.read_csv('trec_health_eval/misinfo-resources-2021/qrels/qrels-35topics.txt', 
                       sep=" ", header=None,
                       names=['qid', 'iter', 'docid', 'rel', 'correct', 'cred'])
print("✅ QRELS loaded!")

print("\ncred distribution:")
print(qrels_df['cred'].value_counts().sort_index().to_string())

✅ QRELS loaded!

cred distribution:
cred
-2      38
-1    6309
 0    2440
 1    3339
 2     652


### Summary statistics for QRELS

In [4]:
print("\n📊 QRELS Summary Table:")
print("="*70)

summary = pd.DataFrame({
    'Metric': [
        'Total Judgments',
        'Unique Queries',
        'Unique Documents',
        'Avg Judgments per Query',
        'Avg cred Score',
        'Avg correct Score', 
        'Avg rel Score'
    ],
    'Value': [
        len(qrels_df),
        qrels_df['qid'].nunique(),
        qrels_df['docid'].nunique(),
        f"{len(qrels_df) / qrels_df['qid'].nunique():.1f}",
        f"{qrels_df['cred'].mean():.2f}",
        f"{qrels_df['correct'].mean():.2f}",
        f"{qrels_df['rel'].mean():.2f}"
    ]
})
print(summary.to_string(index=False))

print("\n📈 Full QRELS Table (first 20 rows):")
print(qrels_df.head(20).to_string(index=False))


📊 QRELS Summary Table:
                 Metric Value
        Total Judgments 12778
         Unique Queries    35
       Unique Documents 12675
Avg Judgments per Query 365.1
         Avg cred Score -0.14
      Avg correct Score  0.22
          Avg rel Score  0.69

📈 Full QRELS Table (first 20 rows):
 qid  iter                                     docid  rel  correct  cred
 101     0  en.noclean.c4-train.00000-of-07168.64607    0       -1    -1
 101     0 en.noclean.c4-train.00005-of-07168.143426    1        2     0
 101     0  en.noclean.c4-train.00011-of-07168.21494    0       -1    -1
 101     0  en.noclean.c4-train.00029-of-07168.37185    0       -1    -1
 101     0  en.noclean.c4-train.00033-of-07168.36297    0       -1    -1
 101     0  en.noclean.c4-train.00039-of-07168.36640    0       -1    -1
 101     0 en.noclean.c4-train.00042-of-07168.139761    0       -1    -1
 101     0  en.noclean.c4-train.00044-of-07168.58816    1        2     0
 101     0 en.noclean.c4-train.00057-of-07

### FILTER VALID DATA 

1. rel = 0? → STOP (-1 flag) supportiveness = -1, credibility = -1 (not useful → skip)
2. rel ≥ 1? → Assess:
    - supportiveness (0 = wrong, 1 = partial, 2 = correct)
    - credibility (0 = low, 2 = PubMed-level)
3. Error?   → supportiveness = -2, credibility = -2 (assessor mistake)

In [5]:
valid_docs = qrels_df[
    (qrels_df['rel'] >= 1) &                    # Must be useful
    (qrels_df['correct'].between(0, 2)) &   # Valid correct
    (qrels_df['cred'].between(0, 2))     # Valid cred
]

print("Filtering results:")
print(f"All entries: {len(qrels_df)}")
print(f"Valid docs: {len(valid_docs)}")
print("\nValid correct distribution:")
print(valid_docs['correct'].value_counts().sort_index())


Filtering results:
All entries: 12778
Valid docs: 6400

Valid correct distribution:
correct
0     884
1    1863
2    3653
Name: count, dtype: int64


### 2. TOPICS (health queries)

In [6]:
import xml.etree.ElementTree as ET
tree = ET.parse('trec_health_eval/misinfo-resources-2021/topics/misinfo-2021-topics.xml')
root = tree.getroot()

topics = []
for topic in root.findall('.//topic'):
    topics.append({
        'qid': topic.find('number').text,
        'query': topic.find('query').text
    })
topics_df = pd.DataFrame(topics)
print("\n✅ TOPICS loaded!")
print("Sample:", topics_df.head(1)['query'].values[0])


✅ TOPICS loaded!
Sample: ankle brace achilles tendonitis


In [7]:
from collections import defaultdict
import re

def parse_and_group(docids):
    groups = defaultdict(list)
    pattern = r'en\.noclean\.c4-train\.(\d{5})-of-\d+\.(\d+)'
    
    for docid in docids:
        match = re.match(pattern, docid)
        if match:
            shard_str, doc_str = match.groups()
            shard = f"{int(shard_str):05d}"
            doc_idx = int(doc_str)
            groups[shard].append(doc_idx)
    
    # FIXED: Create [(key, value), ...] tuples explicitly
    sorted_items = sorted({k: sorted(v) for k, v in groups.items()}.items(), 
                         key=lambda x: int(x[0]))
    return dict(sorted_items)  # Now works: items() gives 2-tuples


In [15]:
relevant_docids = parse_and_group(valid_docs['docid'].unique().tolist())
doc_cred_map = dict(zip(valid_docs['docid'], valid_docs['cred']))
keys = list(relevant_docids.keys())
print(len(relevant_docids), keys[0], keys[-1])
print("Number of docs: ", len(valid_docs['docid'].unique().tolist()))

4243 00000 07167
Number of docs:  6366


### 3. Construct a subset of the C4 collection containing only our relevant_docids

In [ ]:
import gzip
import json
import requests
from pathlib import Path

def extract_c4_docs(shard_groups, cache_dir="c4_shards", output_file="extracted_docs.jsonl"):
    """Extracts only specified docs from shards, minimal storage."""
    Path(cache_dir).mkdir(exist_ok=True)
    extracted = []
    
    with open(output_file, 'a', encoding='utf-8') as out_f:
        for shard_key, doc_idxs in shard_groups.items():
            shard_file = Path(cache_dir) / f"c4-train.{shard_key}-of-07168.json.gz"
            
            # Skip if already cached
            if not shard_file.exists():
                url = f"https://huggingface.co/datasets/allenai/c4/resolve/main/en.noclean/c4-train.{shard_key}-of-07168.json.gz"
                print(f"Downloading {shard_key}...")
                shard_file.write_bytes(requests.get(url).content)
                
                # Extract ONLY target docs
                found_docs = set(doc_idxs)
                with gzip.open(shard_file, "rt", encoding="utf-8") as f:
                    for i, line in enumerate(f):
                        if i in doc_idxs:
                            doc = json.loads(line)
                            doc_id = f'en.noclean.c4-train.{shard_key}-of-07168.{i}'
                            doc_data ={
                                'docid': doc_id,
                                'text': doc['text'],
                                'url': doc['url'],
                                'timestamp': doc['timestamp'],
                                'cred': doc_cred_map.get(doc_id, 1)
                            }
                            out_f.write(json.dumps(doc_data) + '\n')
                            out_f.flush()  # Force write
                            found_docs.discard(i)
                            if not found_docs:
                                break
                # Delete shard
                shard_file.unlink(missing_ok=True)
                # print(f"✓ Shard {shard_key} done + deleted")
    print(f"All docs saved to {output_file}")
                    

In [ ]:
docs = extract_c4_docs(relevant_docids)
print(f"Extracted {len(docs)} docs, shards cached: {len(relevant_docids)}")


### Patch for absent data

In [32]:
# Patching
import json
with open('extracted_docs.jsonl', 'r') as f:
    lines = f.readlines()
    first_line_shard_key = json.loads(lines[0])['docid'].split('.')[3]
    last_line_shard_key = json.loads(lines[-1])['docid'].split('.')[3]

import gzip
import json
import requests
from pathlib import Path

def extract_c4_docs_patched(shard_groups, cache_dir="c4_shards", output_file="extracted_docs.jsonl"):
    """Extracts only specified docs from shards, minimal storage."""
    Path(cache_dir).mkdir(exist_ok=True)
    prepend_file = output_file + ".prepend"
    
    with open(prepend_file, 'w', encoding='utf-8') as prepend_f:
        prepend_f.write("")  # Initialize empty prepend file
    
    for shard_key, doc_idxs in shard_groups.items():
        if shard_key < first_line_shard_key:
            # Process for prepending
            write_file = prepend_file
        elif first_line_shard_key <= shard_key <= last_line_shard_key:
            # Skip - already in the file
            continue
        else:
            # Process for normal append
            write_file = output_file
        
        shard_file = Path(cache_dir) / f"c4-train.{shard_key}-of-07168.json.gz"
        
        # Skip if already cached
        if not shard_file.exists():
            url = f"https://huggingface.co/datasets/allenai/c4/resolve/main/en.noclean/c4-train.{shard_key}-of-07168.json.gz"
            print(f"Downloading {shard_key}...")
            shard_file.write_bytes(requests.get(url).content)
            
            # Extract ONLY target docs
            found_docs = set(doc_idxs)
            with gzip.open(shard_file, "rt", encoding="utf-8") as f:
                with open(write_file, 'a', encoding='utf-8') as out_f:
                    for i, line in enumerate(f):
                        if i in doc_idxs:
                            doc = json.loads(line)
                            doc_id = f'en.noclean.c4-train.{shard_key}-of-07168.{i}'
                            doc_data = {
                                'docid': doc_id,
                                'text': doc['text'],
                                'url': doc['url'],
                                'timestamp': doc['timestamp'],
                                'cred': doc_cred_map.get(doc_id, 1)
                            }
                            out_f.write(json.dumps(doc_data) + '\n')
                            out_f.flush()  # Force write
                            found_docs.discard(i)
                            if not found_docs:
                                break
            # Delete shard
            shard_file.unlink(missing_ok=True)
    
    # Merge prepend file with main output file
    prepend_path = Path(prepend_file)
    if prepend_path.exists() and prepend_path.stat().st_size > 0:
        # Read prepend and main files
        with open(prepend_file, 'r', encoding='utf-8') as f:
            prepend_content = f.read()
        
        existing_content = ""
        if Path(output_file).exists():
            with open(output_file, 'r', encoding='utf-8') as f:
                existing_content = f.read()
        
        # Write prepend + main to output file
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(prepend_content)
            f.write(existing_content)
        
        prepend_path.unlink()  # Delete temp prepend file
    
    print(f"All docs saved to {output_file}")

In [34]:
extract_c4_docs_patched(relevant_docids)

All docs saved to extracted_docs.jsonl
